In [ ]:
# Hive Dynamic Partitions
#   Tables are nothing but files i.e., tables are stored as files at the backend.
#   When you want to insert data from files to partitioned dynamic table its does not directly possible.
#   We need convert file into table as it is and it is called stage table and then need to load stage table to partitioned dynamic table


In [9]:
%%sql
use spark_catalog

StatementMeta(, dad6718c-a0db-4fd1-b695-99c0e13d48c6, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [10]:
%%sql

create database if not exists db_hymaa_stage
location 'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/'


StatementMeta(, dad6718c-a0db-4fd1-b695-99c0e13d48c6, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
use db_hymaa_stage

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
#Convert new data file into datafrme
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

population_schema=StructType([
    StructField('pop_id',StringType(),True),
    StructField('pop_name',StringType(),True),
    StructField('pop_salary',IntegerType(),True),
    StructField('pop_gender',StringType(),True),
    StructField('pop_age',IntegerType(),True),
    StructField('pop_state',StringType(),True)
])

df=spark.read.csv(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/Popuation/population_2.txt',
    header=False,
    schema=population_schema,
    sep='\t'
)

display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26905817-1c77-474b-ad7c-dfb861db7e59)

In [4]:
#Create a new table for maintaining bronze layer and append latest data to partitioned table
df.write.format('delta').mode('append').saveAsTable('population_stage')
df.write.format('delta').mode('append').partitionBy('pop_state','pop_gender').saveAsTable('db_hymaa_test.population_dynamic_multi')

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 6, Finished, Available, Finished, False)

In [15]:
%%sql
--see data in both the tables
select * from population_stage;
select * from db_hymaa_test.population_dynamic_multi;

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 30, Finished, Available, Finished, True)

<Spark SQL result set with 6 rows and 6 fields>

<Spark SQL result set with 12 rows and 6 fields>

In [16]:
%%sql
--Check for partitions - SHow partition command does not work in fabric notebook if table is not partitioned
show partitions db_hymaa_test.population_dynamic_multi;
show partitions db_hymaa_stage.population_stage;

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 32, Finished, , Finished, True)

<Spark SQL result set with 5 rows and 1 fields>

Error: [INVALID_PARTITION_OPERATION.PARTITION_SCHEMA_IS_EMPTY] The partition command is invalid. Table `spark_catalog`.`chimcobldhq2aq3pdlgm2nrge9gm6t39cdiluchg68r2ar38btp62tp5chh5uq3pdlgm2nrjehgmep8`.`population_stage` is not partitioned.; line 2 pos 16;
ShowPartitions [partition#15443]
+- ResolvedTable org.apache.spark.sql.delta.catalog.DeltaCatalog@69110505, chimcobldhq2aq3pdlgm2nrge9gm6t39cdiluchg68r2ar38btp62tp5chh5uq3pdlgm2nrjehgmep8.population_stage, DeltaTableV2(org.apache.spark.sql.SparkSession@1d2c48fa,abfss://9d217522-839e-4c2d-a44d-cbc79aedf1ad@onelake.dfs.fabric.microsoft.com/3fce7576-a611-4faf-9a52-bd49846b439b/Tables/db_hymaa_stage/population_stage,Some(CatalogTable(
Catalog: spark_catalog
Database: `hymaa_practice_2026`.`lh_raw`.`db_hymaa_stage`
Table: population_stage
Created Time: Wed Jan 21 14:03:23 UTC 1970
Last Access: UNKNOWN
Created By: Spark 
Type: MANAGED
Provider: delta
Comment: Delta table auto-discovered from Onelake: 3fce7576-a611-4faf-9a52-bd49846b439b/Tab

In [9]:
%%sql
--Different ways to check whether table is partitioned or not  - Observe the difference
desc formatted db_hymaa_test.population_dynamic_multi;
desc formatted population_stage;

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 14, Finished, Available, Finished, True)

<Spark SQL result set with 17 rows and 3 fields>

<Spark SQL result set with 13 rows and 3 fields>

In [13]:
%%sql
--Different ways to check whether table is partitioned or not  - Observe the difference
DESCRIBE DETAIL db_hymaa_test.population_dynamic_multi;
DESCRIBE TABLE db_hymaa_test.population_dynamic_multi;
DESCRIBE DETAIL db_hymaa_stage.population_stage;
DESCRIBE TABLE db_hymaa_stage.population_stage;

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 26, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 15 fields>

<Spark SQL result set with 10 rows and 3 fields>

<Spark SQL result set with 1 rows and 15 fields>

<Spark SQL result set with 6 rows and 3 fields>

In [19]:
#JSON file processing - Databricks and Fabric syntax is same

#JSON With single line
#Input - {"id":101,"name":"srinivas","gender":"male","language":"english","location":"hyderabad"}

df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_0.json')
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f560ee18-8c1f-48f7-b5c3-d5f6927d52c8)

In [21]:
#JSON file processing

#JSON With multi line
# Input - {
# "id":"1",
# "name":"srinivas",
# "gender":"male",
# "language":"english",
# "location":"hyderabad"
# }


df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_1.json',multiLine=True)
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8d02aac4-05e0-40b3-b1b2-a627aa5804c7)

In [22]:
#JSON file processing

#JSON With multi line - Multiple records
# Input - 
# [
# {
# "id":"101",
# "name":"srinivas",
# "gender":"male"
# },
# {
# "id":"102",
# "name":"phani",
# "gender":"male"
# }
# ]


df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_2.json',multiLine=True)
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e079bf27-a62f-4119-a205-ef8b0afcb6b3)

In [23]:
#JSON file processing

#JSON With multi line - Multiple records - With missing columns
# Input - 
# [
# {
# "id":"101",
# "name":"srinivas",
# "gender":"male",
# "phonenumber":9999
# },
# {
# "id":"102",
# "name":"phani",
# "gender":"male",
# "location":"Hyderabad"
# }
# ]


df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_3.json',multiLine=True)
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26106f92-834f-47df-8af5-68a4853785ea)

In [24]:
#JSON file processing

#JSON With multi line - Multiple records - With nested data 
# Input - 
# [
# {
# "id":"101",
# "name":"srinivas",
# "gender":"male",
# "address":{
#             "city":"Hyderabad",
# 			"country":"india"
#           } 
# },
# {
# "id":"102",
# "name":"phani",
# "gender":"male",
# "address":{
#             "city":"Hyderabad",
# 			"country":"india"
#           }
# }
# ]


df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_4.json',multiLine=True)
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a512b094-6ab9-44d4-a663-116150dc8ad1)

In [32]:
#Convert above table dataframe into seperate columns data
df1=df.select('address.city','address.country','gender','id','name')
display(df1)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4c730e4c-433d-46c6-bacb-5bb131d76538)

In [33]:
#JSON file processing

#JSON With multi line - Multiple records - With nested and nested data 
# Input - 
# [
# {
# "id":"101",
# "name":"srinivas",
# "gender":"male",
# "address":{
#             "city":"Hyderabad",
# 			"country":"india",
# 			"house_dtls":{
# 			                "doorno":"1-2-3",
# 							"street":"srnagar"
# 			             }
#           } 
# },
# {
# "id":"102",
# "name":"phani",
# "gender":"male",
# "address":{
#             "city":"Hyderabad",
# 			"country":"india",
# 			"house_dtls":{
# 			                "doorno":"1-2-3",
# 							"street":"ameerpet"
# 			             }
#           }
# }
# ]


df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_5.json',multiLine=True)
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ac6b7187-655e-411f-afcc-1a687a448079)

In [34]:
#Convert above table dataframe into seperate columns data
df1=df.select('address.city','address.country','address.house_dtls.doorno','address.house_dtls.street','gender','id','name')
display(df1)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37025ae1-90be-4be6-90d2-b89c529ac06a)

In [35]:
#JSON file processing

#JSON With multi line - Multiple records - With array of values
# Input - 
# [
# {
# "id":"101",
# "name":"srinivas",
# "gender":"male",
# "languages":["English","Hindi"]
# },
# {
# "id":"102",
# "name":"phani",
# "gender":"male",
# "languages":["English","Hindi"]
# }
# ]



df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_6.json',multiLine=True)
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c4116cc2-36a3-4113-8a1a-a54fbf60783b)

In [37]:
from pyspark.sql.functions import explode

df1=df.select('gender','id','name',explode('languages'))
display(df1)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 15b18323-f999-42c9-9d5b-49d316d4daf1)

In [46]:
#JSON file processing

#JSON With multi line - Multiple records - With array of  and nested data
# Input - 
# [
# {
# "id":"101",
# "name":"srinivas",
# "gender":"male",
# "languages":["English","Hindi"],
# "address":{
#             "city":"Hyderabad",
# 			"country":"india"
#           } 
# },
# {
# "id":"102",
# "name":"phani",
# "gender":"male",
# "languages":["English","Hindi"],
# "address":{
#             "city":"Hyderabad",
# 			"country":"india"
#           }
# }
# ]





df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_7.json',
    multiLine=True)
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 62, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3bd9d6ce-e14a-424e-abdc-5afadce5da9a)

In [47]:
df1=df.select('address.city','address.country','gender','id',explode('languages').alias('lang'),'name')
display(df1)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ed58e88a-16da-4251-bcd9-abe9a315d012)

In [48]:
#JSON file processing

#JSON With multi line - Multiple records - With corrupted data
# Input - 
# [
# {
# "id":"101",
# "name":"srinivas"
# "gender":"male"
# },
# {
# "id":"102",
# "name":"phani",
# "gender":"male"
# }
# ]





df=spark.read.json(
    'abfss://hymaa_practice_2026@onelake.dfs.fabric.microsoft.com/lh_raw.Lakehouse/Files/spark json files/emp_details_corrupted.json',multiLine=True)
display(df)

StatementMeta(, 3e854270-6796-42a0-89e8-7d22e78817cc, 64, Finished, Available, Finished, False)

AnalysisException: Since Spark 2.3, the queries from raw JSON/CSV files are disallowed when the
referenced columns only include the internal corrupt record column
(named _corrupt_record by default). For example:
spark.read.schema(schema).csv(file).filter($"_corrupt_record".isNotNull).count()
and spark.read.schema(schema).csv(file).select("_corrupt_record").show().
Instead, you can cache or save the parsed results and then send the same query.
For example, val df = spark.read.schema(schema).csv(file).cache() and then
df.filter($"_corrupt_record".isNotNull).count().